# Laboratorium 5 - rekomendacje grupowe

## Przygotowanie

 * pobierz i wypakuj dataset: https://files.grouplens.org/datasets/movielens/ml-latest-small.zip
   * więcej możesz poczytać tutaj: https://grouplens.org/datasets/movielens/
 * [opcjonalnie] Utwórz wirtualne środowisko
 `python3 -m venv ./recsyslab5`
 * zainstaluj potrzebne biblioteki:
 `pip install numpy pandas scipy matplotlib`

## Część 1. - przygotowanie danych

In [18]:
# importujemy wszystkie potrzebne pakiety

import math
import numpy as np
import pandas as pd
from scipy.sparse.linalg import svds


from random import choice, sample
from statistics import mean, stdev

In [19]:
PATH = 'ml-latest-small'

In [20]:
# wczytujemy oceny uzytkownikow i obliczamy (za pomoc dekompozycji macierzy) wszystkie przewidywane oceny filmow

def read_ratings(path, k=600, scale_factor=2.0, print_stats=True):
    # idea: https://www.kaggle.com/code/indralin/movielens-project-1-2-collaborative-filtering
    reviews = pd.read_csv(f'{path}/ratings.csv', names=['userId', 'movieId', 'rating', 'time'], delimiter=',', engine='python', skiprows=1)
    
    reviews.drop(['time'], axis=1, inplace=True)
    reviews_no, _ = reviews.shape
    reviews_matrix = reviews.pivot(index='userId', columns='movieId', values='rating')
    movies = reviews_matrix.columns
    users = reviews_matrix.index
    users_no, movies_no = reviews_matrix.shape
    print(f'Got {reviews_no} reviews for {movies_no} movies and {users_no} users.')

    user_ratings_mean = np.nanmean(reviews_matrix.values, axis=1)
    normalized_reviews_matrix = np.nan_to_num(reviews_matrix.values - user_ratings_mean.reshape(-1, 1), 0.0)

    U, sigma, Vt = svds(normalized_reviews_matrix, k=k)
    sigma = np.diag(sigma)
    predicted_ratings = np.dot(np.dot(U, sigma), Vt) + user_ratings_mean.reshape(-1, 1).clip(0.5, 5.0)
    mean_square_error = np.nanmean(np.square(predicted_ratings - reviews_matrix.values))
    std_square_error = np.nanstd(np.square(predicted_ratings - reviews_matrix.values))
    print(f'Reviews prediction mean square error = {mean_square_error}')
    print(f'Reviews prediction standatd deviation of square error = {std_square_error}')

    if print_stats:
        stats = [
            ('metric', 'dataset', 'prediction'),
            ('avg', np.nanmean(reviews_matrix), np.mean(predicted_ratings)),
            ('st_dev', np.nanstd(reviews_matrix), np.std(predicted_ratings)),
            ('median', np.nanmedian(reviews_matrix), np.median(predicted_ratings)),
            ('p25', np.nanquantile(reviews_matrix, 0.25), np.quantile(predicted_ratings, 0.25)),
            ('p75', np.nanquantile(reviews_matrix, 0.75), np.quantile(predicted_ratings, 0.75))
        ]
        print('Stats (for raings in original range [0.5, 5.0]):')
        print('\n'.join([str(s) for s in stats]))

    rounded_predictions = np.rint(scale_factor * predicted_ratings) # cast values to {1, 2, ..., 10}
    return pd.DataFrame(data=rounded_predictions, index=list(users), columns=list(movies))
    
ratings = read_ratings(PATH)
# dostep do danych:
# ratings[movieId][userId] pobiera 1 wartosc
# ratings.loc[:, movieId] pobiera wektor dla danego filmu
# ratings.loc[userId, :] pobiera wektor dla danego uzytkownika
ratings

Got 100836 reviews for 9724 movies and 610 users.
Reviews prediction mean square error = 1.6577787842924333e-05
Reviews prediction standatd deviation of square error = 0.0007928950536518298
Stats (for raings in original range [0.5, 5.0]):
('metric', 'dataset', 'prediction')
('avg', np.float64(3.501556983616962), np.float64(3.6572223377474))
('st_dev', np.float64(1.0425240696180562), np.float64(0.4954623756097101))
('median', np.float64(3.5), np.float64(3.7052240088117694))
('p25', np.float64(3.0), np.float64(3.357451718316401))
('p75', np.float64(4.0), np.float64(3.9999816268830006))
Reviews prediction mean square error = 1.6577787842924333e-05
Reviews prediction standatd deviation of square error = 0.0007928950536518298
Stats (for raings in original range [0.5, 5.0]):
('metric', 'dataset', 'prediction')
('avg', np.float64(3.501556983616962), np.float64(3.6572223377474))
('st_dev', np.float64(1.0425240696180562), np.float64(0.4954623756097101))
('median', np.float64(3.5), np.float64(3.

,1,2,3,4,5,6,7,8,9,10,...,193565,193567,193571,193573,193579,193581,193583,193585,193587,193609
1,8.0,9.0,8.0,9.0,9.0,8.0,9.0,9.0,9.0,9.0,...,9.0,9.0,9.0,9.0,9.0,9.0,9.0,9.0,9.0,9.0
2,8.0,8.0,8.0,8.0,8.0,8.0,8.0,8.0,8.0,8.0,...,8.0,8.0,8.0,8.0,8.0,8.0,8.0,8.0,8.0,8.0
3,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,...,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0
4,7.0,7.0,7.0,7.0,7.0,7.0,7.0,7.0,7.0,7.0,...,7.0,7.0,7.0,7.0,7.0,7.0,7.0,7.0,7.0,7.0
5,8.0,7.0,7.0,7.0,7.0,7.0,7.0,7.0,7.0,7.0,...,7.0,7.0,7.0,7.0,7.0,7.0,7.0,7.0,7.0,7.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
606,5.0,7.0,7.0,7.0,7.0,7.0,5.0,7.0,7.0,7.0,...,7.0,7.0,7.0,7.0,7.0,7.0,7.0,7.0,7.0,7.0
607,8.0,8.0,8.0,8.0,8.0,8.0,8.0,8.0,8.0,8.0,...,8.0,8.0,8.0,8.0,8.0,8.0,8.0,8.0,8.0,8.0
608,5.0,4.0,4.0,6.0,6.0,6.0,6.0,6.0,6.0,8.0,...,6.0,6.0,6.0,6.0,6.0,6.0,6.0,6.0,6.0,6.0
609,6.0,7.0,7.0,7.0,7.0,7.0,7.0,7.0,7.0,8.0,...,7.0,7.0,7.0,7.0,7.0,7.0,7.0,7.0,7.0,7.0


In [21]:
# wczytujemy nazwy filmow i kategorie

movies_metadata = pd.read_csv('ml-latest-small/movies.csv').set_index('movieId')
movies_metadata

,title,genres
movieId,,
1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
2,Jumanji (1995),Adventure|Children|Fantasy
3,Grumpier Old Men (1995),Comedy|Romance
4,Waiting to Exhale (1995),Comedy|Drama|Romance
5,Father of the Bride Part II (1995),Comedy
...,...,...
193581,Black Butler: Book of the Atlantic (2017),Action|Animation|Comedy|Fantasy
193583,No Game No Life: Zero (2017),Animation|Comedy|Fantasy
193585,Flint (2017),Drama


In [22]:
# wczytujemy przykladowe grupy uzytkownikow
groups = pd.read_csv('groups.csv').values.tolist()
groups

[[111, 307, 474, 599, 414],
 [469, 182, 232, 448, 600],
 [508, 581, 497, 402, 566],
 [300, 515, 245, 568, 507],
 [2, 371, 252, 518, 37],
 [269, 360, 469, 287, 308],
 [243, 527, 418, 118, 370],
 [186, 559, 327, 553, 314]]

In [23]:
# przygotowujemy funkcje pomocnicza

def describe_group(group, N=10):
    print(f'\n\nUser ids: {group}')
    group_size = len(group)
    
    mean_stdev = ratings.loc[group].std(axis=0).mean()
    median_stdev = ratings.loc[group].std(axis=0).median()
    std_stdev = ratings.loc[group].std(axis=0).std()
    print(f'\nMean ratings deviation: {mean_stdev}')
    print(f'Median ratings deviation: {median_stdev}')
    print(f'Standard deviation of ratings deviation: {std_stdev}')
    
    average_scores = ratings.iloc[group].mean(axis=0)
    average_scores = average_scores.sort_values()
    best_movies = [(movies_metadata['title'][movie_id], average_scores[movie_id]) for movie_id in list(average_scores[-N:].index)]
    worst_movies = [(movies_metadata['title'][movie_id], average_scores[movie_id]) for movie_id in list(average_scores[:N].index)]
    
    print('\nBest movies:')
    for movie, score in best_movies[::-1]:
        print(f'{movie}, {score}*')
    print('\nWorst movies:')
    for movie, score in worst_movies:
        print(f'{movie}, {score}*')

describe_group(groups[5])



User ids: [269, 360, 469, 287, 308]

Mean ratings deviation: 1.125914957457979
Median ratings deviation: 1.0954451150103321
Standard deviation of ratings deviation: 0.17836724055768716

Best movies:
Toy Story (1995), 8.2*
Forrest Gump (1994), 8.2*
Willy Wonka & the Chocolate Factory (1971), 8.0*
Braveheart (1995), 8.0*
Terminator 2: Judgment Day (1991), 7.8*
Schindler's List (1993), 7.8*
Dances with Wolves (1990), 7.6*
James and the Giant Peach (1996), 7.6*
Dead Man Walking (1995), 7.6*
Nixon (1995), 7.6*

Worst movies:
Broken Arrow (1996), 5.2*
Sleepy Hollow (1999), 5.4*
The Devil's Advocate (1997), 5.4*
Cable Guy, The (1996), 5.4*
Mission: Impossible (1996), 5.6*
Nutty Professor, The (1996), 5.6*
Down Periscope (1996), 5.8*
Cheech and Chong's Up in Smoke (1978), 5.8*
Wrong Man, The (1956), 5.8*
Fog, The (2005), 5.8*


## Część 2. - algorytmy proste

In [24]:
# zdefiniujmy interfejs dla wszystkich algorytmow rekomendacyjnych

class Recommender:
    def recommend(self, movies, ratings, group, size):
        pass


# jako pierwszy zaimplementujemy algorytm losowy - dla porownania
    
class RandomRecommender(Recommender):
    def __init__(self):
        self.name = 'random'
        
    def recommend(self, movies, ratings, group, size):
        return sample(movies, size)

In [25]:
# algorytm rekomendujacy filmy o najwyzszej sredniej ocen

class AverageRecommender(Recommender):
    def __init__(self):
        self.name = 'average'
    
    def recommend(self, movies, ratings, group, size):
        return list(ratings.loc[group].mean(axis=0).sort_values(ascending=False).index[:size])

In [26]:
# algorytm rekomendujacy filmy o najwyzszej sredniej ocen,
#   ale rownoczesnie wykluczajacy te filmy, ktore otrzymaly choc jedna ocene ponizej thresholdu

class AverageWithoutMiseryRecommender(Recommender):
    def __init__(self, score_threshold):
        self.name = 'average_without_misery'
        self.score_threshold = score_threshold
        
    def recommend(self, movies, ratings, group, size):
        return list(ratings.loc[group].mean(axis=0).sort_values(ascending=False)[ratings.loc[group].min(axis=0) >= self.score_threshold].index[:size])

In [27]:
# algorytm uwzgledniajacy preferencje tylko jednego uzytkownika w kazdej iteracji

class FairnessRecommender(Recommender):
    def __init__(self):
        self.name = 'fairness'
        
    def recommend(self, movies, ratings, group, size):
        recommendations = []
        i = 0
        user_movie_idx = {user_id: 0 for user_id in group}

        while len(recommendations) < size:
            user_id = group[i % len(group)]
            top_movie = ratings.loc[user_id].sort_values(ascending=False).index[user_movie_idx[user_id]]
            if top_movie not in recommendations:
                recommendations.append(top_movie)
            user_movie_idx[user_id] += 1
            i += 1

        return recommendations

In [28]:
# wybrany algorytm wyborczy (dyktatura, Borda, Copeland)

class VotingRecommender(Recommender):
    def __init__(self):
        self.name = 'borda'
    
    def recommend(self, movies, ratings, group, size):
        borda_scores = {movie: 0 for movie in movies}
        n = len(movies)
        
        for user_id in group:
            user_ratings = ratings.loc[user_id]
            relevant_ratings = user_ratings[user_ratings.index.isin(movies)]
            
            grouped_by_rating = relevant_ratings.groupby(relevant_ratings).apply(lambda x: x.index.tolist())
            sorted_groups = grouped_by_rating.sort_index(ascending=False)
            
            current_points =  - 1
            
            for rating_val, movie_ids in sorted_groups.items():
                group_size = len(movie_ids)
                
                for movie_id in movie_ids:
                    borda_scores[movie_id] += current_points
                
                current_points -= group_size
        
        sorted_movies = sorted(borda_scores.items(), key=lambda x: x[1], reverse=True)
        return [movie_id for movie_id, score in sorted_movies[:size]]

In [29]:
# algorytm zachlanny, aproksymujacy metode Proportional Approval Voting
#   w kazdej iteracji wybieramy ten film, ktory najbardziej zwieksza zadowolenie zgodnie z punktacja PAV

class ProportionalApprovalVotingRecommender(Recommender):
    def __init__(self, threshold):
        self.threshold = threshold
        self.name = 'PAV'

    def recommend(self, movies, ratings, group, size):
        satisfaction = {user: 0 for user in group}
        recommendation = []
        for _ in range(size):
            best_movie = None
            best_increase = -math.inf
            for movie in movies:
                increase = 0
                for user in group:
                    if ratings.at[user, movie] >= self.threshold:
                        increase += 1 / (satisfaction[user] + 1)
                if increase > best_increase:
                    best_increase = increase
                    best_movie = movie
            recommendation.append(best_movie)
            for user in group:
                if ratings.at[user, best_movie] >= self.threshold:
                    satisfaction[user] += 1
            movies.remove(best_movie)
        return recommendation

## Część 3. - funkcje celu

In [30]:
# dwie funkcje pomocnicze:
#  - znajdujaca ulubione filmy danego uzytkownika
#  - obliczajaca sume ocen wystawionych przez uzytkownika wszystkim filmom w rekomendacji

def top_n_movies_for_user(ratings, movies, user_id, n):
    user_ratings = ratings.loc[user_id, movies]
    top_movies = user_ratings.sort_values(ascending=False).index[:n]
    return list(top_movies)

def total_score(recommendation, user_id, ratings):
    user_ratings = ratings.loc[user_id, recommendation]
    return user_ratings.sum()

In [31]:
# funkcja obliczajaca zadowolenie pojedynczego uzytkownika
#  - iloraz zadowolenia z wygenerowanej rekomendacji oraz zadowolenia z hipotetycznej rekomendacji idealnej
def overall_user_satisfaction(recommendation, user_id, movies, ratings):
    rec_score = total_score(recommendation, user_id, ratings)
    ideal_score = total_score(list(top_n_movies_for_user(ratings, movies, user_id, len(recommendation))), user_id, ratings)
    if ideal_score == 0:
        return 0
    return rec_score / ideal_score

# funkcja celu - srednia z zadowolenia wszystkich uzytkownikow w grupie
def overall_group_satisfaction(recommendation, group, movies, ratings):
    satisfactions = [overall_user_satisfaction(recommendation, user_id, movies, ratings) for user_id in group]
    return sum(satisfactions) / len(satisfactions) if satisfactions else 0

# funkcja celu - roznica miedzy maksymalnym i minimalnym zadowolenie w grupie
def group_disagreement(recommendation, group, movies, ratings):
    satisfactions = [overall_user_satisfaction(recommendation, user_id, movies, ratings) for user_id in group]
    if not satisfactions:
        return 0
    return max(satisfactions) - min(satisfactions)

## Część 4. - Sequential Hybrid Aggregation

In [32]:
# algorytm balansujacy pomiedzy wyborem elementow o najwyzszej sredniej ocen
#   i o najwyzszej minimalnej ocenie
#   wyliczajacy w kazdej iteracji parametr alfa - jak na wykladzie
class SequentialHybridAggregationRecommender(Recommender):
    def __init__(self):
        self.name = 'sequential_hybrid_aggregation'

    def recommend(self, movies, ratings, group, size):
        alfa = 0.0
        group_movies = ratings.loc[group, movies]
        means = group_movies.mean(axis=0)
        mins = group_movies.min(axis=0)

        recommendation = []
        for _ in range(size):
            movie_scores = {
                movie: (1-alfa) * means[movie] + alfa * mins[movie] for movie in group_movies
            }
            best_movie = max(movie_scores, key=movie_scores.get)
            recommendation.append(best_movie)
            alfa = group_disagreement(recommendation, group, movies, ratings)
            group_movies = group_movies.drop(columns=[best_movie])
        return recommendation

## Część 5. - porównanie algorytmów

In [33]:
recommenders = [
    RandomRecommender(),
    AverageRecommender(),
    AverageWithoutMiseryRecommender(5),
    FairnessRecommender(),
    VotingRecommender(),
    ProportionalApprovalVotingRecommender(7),
    SequentialHybridAggregationRecommender()
]

recommendation_size = 10

movies = ratings.columns.tolist()

# dla kazdego algorytmu:
#  - wygenerujmy jedna rekomendacje dla kazdej grupy
#  - obliczmy wartosci obu funkcji celu dla kazdej rekomendacji
#  - wypiszmy wyniki na konsole

for recommender in recommenders:
    print(f'\n\n Recommender: {recommender.name}')
    group_satisfaction = []
    group_disagreements = []
    for test_group in groups:
        recommendation = recommender.recommend(
            movies, ratings, test_group, recommendation_size)
        group_satisfaction.append(overall_group_satisfaction(
            recommendation, test_group, movies, ratings))
        group_disagreements.append(group_disagreement(
            recommendation, test_group, movies, ratings))
    
    print(
        f'Group satisfaction: {mean(group_satisfaction):.4f} +- {stdev(group_satisfaction):.4f}')
    print(
        f'Group disagreement: {mean(group_disagreements):.4f} +- {stdev(group_disagreements):.4f}')



 Recommender: random
Group satisfaction: 0.7436 +- 0.0900
Group disagreement: 0.2147 +- 0.0592


 Recommender: average
Group satisfaction: 0.8750 +- 0.0423
Group disagreement: 0.1806 +- 0.0725


 Recommender: average_without_misery
Group satisfaction: 0.8701 +- 0.0465
Group disagreement: 0.1923 +- 0.0839


 Recommender: fairness
Group satisfaction: 0.8125 +- 0.0701
Group disagreement: 0.1807 +- 0.0561


 Recommender: borda
Group satisfaction: 0.8613 +- 0.0443
Group disagreement: 0.2117 +- 0.1050


 Recommender: PAV
Group satisfaction: 0.8367 +- 0.0552
Group disagreement: 0.2328 +- 0.1895


 Recommender: sequential_hybrid_aggregation
Group satisfaction: 0.8744 +- 0.0509
Group disagreement: 0.1874 +- 0.0784
